# Publications HAL par laboratoire (2015–2025)

Interrogation de l'API HAL (`https://api.archives-ouvertes.fr/search/`) par `structId_i` pour chaque laboratoire.

In [ ]:
import requests
import pandas as pd
import time
from tqdm.notebook import tqdm

## Liste des laboratoires

In [ ]:
LABS = [
    ("CMP",           {244423}),
    ("CRAN",          {185180, 1001}),
    ("CREATIS",       {139739}),
    ("CRIL",          {90448, 1628}),
    ("CRISTAL",       {410272, 389110, 388977, 390300, 183073, 111636, 24885, 2546, 186929}),
    ("DI ENS",        {25027}),
    ("CROSSING (IRL)",{1063106}),
    ("ETIS",          {1003474, 1061575, 1087906, 1003348}),
    ("FILOFOCS (UMI)",{1006288, 1006289}),
    ("GIPSA-Lab",     {1043333, 1042376, 24470}),
    ("GREYC",         {150}),
    ("G-SCOP",        {1043137, 1041927, 74240}),
    ("HEUDIASYC",     {389870}),
    ("I3S",           {13009, 552896, 1079434}),
    ("ICUBE",         {217648, 1073080}),
    ("IDRIS",         {1823}),
    ("IPAL (IRL)",    {542003, 220880, 138926}),
    ("IRIF",          {1005016, 444497}),
    ("IRISA",         {491183, 491231, 490899, 491188, 1092618, 491177, 1092619, 490900,
                       419364, 419370, 105128, 2494, 25255, 419365, 419367, 491230,
                       419363, 419366, 491232, 419362, 1099404, 545024, 1099406,
                       1099401, 1099402, 1099435, 525233, 1088566, 1088569, 495900,
                       489780, 1092631, 1092630, 1092628, 1092632, 1092626, 1092625, 1092629}),
    ("IRIT",          {34499, 1082335}),
    ("ISIR",          {541937, 96164}),
    ("JFLI",          {542009, 229050}),
    ("L2S",           {1051117, 1289}),
    ("LAAS",          {459}),
    ("LABRI",         {3102}),
    ("LAB-STICC",     {486345, 491660, 199324, 81533, 1089048}),
    ("LAMIH",         {1067790, 1303}),
    ("LAMSADE",       {989}),
    ("LIG",           {1043301, 1041964, 24471}),
    ("LIGM",          {1001627, 3210}),
    ("LIMOS",         {1063677, 490706, 857}),
    ("LIP",           {35418}),
    ("LIP6",          {541703, 233, 1095103}),
    ("LIPN",          {1000994, 994, 1086916, 1056718}),
    ("LIRIS",         {2003, 1086665}),
    ("LIRMM",         {181, 1071941}),
    ("LIS",           {527033, 199402, 199394, 862, 178374}),
    ("LISN",          {1061259, 1041968, 247329, 2544, 1050003, 81750}),
    ("LIX",           {2071, 1041697, 1071530, 1070048}),
    ("LMF",           {1065710}),
    ("LORIA",         {206040, 466633}),
    ("LS2N",          {1088564, 473973, 95421, 1693, 21439}),
    ("LMF-LSV",       {1065710, 1042689, 2571}),
    ("MDLS",          {210816}),
    ("RELAX",         {1040410, 528907}),
    ("ROOT (IRL)",    {389700}),
    ("STMS",          {541779, 1374}),
    ("TIMA",          {1043043, 1044023, 640}),
    ("TIMC IMAG",     {1043049, 1070489, 1042061, 707, 574002, 555959, 1056575}),
    ("VERIMAG",       {1043148, 1041816, 194}),
]

## Fonction d'interrogation de l'API HAL

In [ ]:
HAL_API_URL = "https://api.archives-ouvertes.fr/search/"
YEAR_MIN    = 2015
YEAR_MAX    = 2025
BATCH_SIZE  = 1000   # maximum accepté par l'API HAL
SLEEP_SEC   = 0.3    # pause entre requêtes pour ne pas surcharger l'API


def fetch_lab_publications(lab_name: str, struct_ids: set,
                           year_min: int = YEAR_MIN,
                           year_max: int = YEAR_MAX) -> list[dict]:
    """
    Récupère toutes les publications HAL d'un laboratoire
    identifié par ses structId_i, entre year_min et year_max.
    Gère la pagination automatiquement.
    Champs récupérés : halId, titre, auteurs, DOI, année, type de doc, langue.
    """
    struct_query = " OR ".join(str(sid) for sid in struct_ids)
    q  = f"structId_i:({struct_query})"
    fq = f"producedDateY_i:[{year_min} TO {year_max}]"

    records = []
    start   = 0

    while True:
        params = {
            "q":     q,
            "fq":    fq,
            "fl":    "halId_s,title_s,authFullName_s,doiId_s,"
                     "producedDateY_i,docType_s,language_s",
            "rows":  BATCH_SIZE,
            "start": start,
            "wt":    "json",
        }

        resp = requests.get(HAL_API_URL, params=params, timeout=60)
        resp.raise_for_status()
        data = resp.json()

        docs      = data["response"]["docs"]
        num_found = data["response"]["numFound"]

        for doc in docs:
            title_field = doc.get("title_s")
            # language_s est une liste dans HAL
            lang_field  = doc.get("language_s")
            records.append({
                "laboratoire": lab_name,
                "auteurs":     "; ".join(doc.get("authFullName_s") or []),
                "titre":       title_field[0] if title_field else None,
                "doi":         doc.get("doiId_s"),
                "annee":       doc.get("producedDateY_i"),
                "type_doc":    doc.get("docType_s"),
                "langue":      lang_field[0] if lang_field else None,
                "hal_id":      doc.get("halId_s"),
            })

        start += len(docs)
        if start >= num_found or not docs:
            break

        time.sleep(SLEEP_SEC)

    return records

## Collecte des données pour tous les laboratoires

In [ ]:
all_records = []
errors      = []

for lab_name, struct_ids in tqdm(LABS, desc="Laboratoires"):
    try:
        records = fetch_lab_publications(lab_name, struct_ids)
        all_records.extend(records)
        print(f"  {lab_name:20s} → {len(records):>5} publications")
    except Exception as e:
        errors.append((lab_name, str(e)))
        print(f"  ⚠ Erreur pour {lab_name}: {e}")

print(f"\nTotal brut : {len(all_records)} enregistrements")
if errors:
    print(f"Laboratoires en erreur : {[e[0] for e in errors]}")

## Construction du dataframe global

In [ ]:
DF_COLS = ["laboratoire", "auteurs", "titre", "doi", "annee", "type_doc", "langue", "hal_id"]

df = pd.DataFrame(all_records, columns=DF_COLS)
df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

print(f"Shape : {df.shape}")
print(f"Années     : {sorted(df['annee'].dropna().unique().tolist())}")
print(f"Labo       : {df['laboratoire'].nunique()}")
print(f"Types doc  : {sorted(df['type_doc'].dropna().unique().tolist())}")
print(f"Langues    : {sorted(df['langue'].dropna().unique().tolist())}")
df.head(10)

## Dédoublonnage + restauration des résultats OpenAlex

Les résultats OpenAlex sont **restaurés depuis le fichier `hal_openalex_uniques.csv`** (déjà calculé) sans relancer les requêtes API.

In [ ]:
# ── Publications uniques ─────────────────────────────────────────────────────
df_unique = df.drop_duplicates(subset=["hal_id"]).reset_index(drop=True)
print(f"Publications uniques : {len(df_unique)}")

# ── Restauration des résultats OpenAlex depuis le CSV existant ───────────────
OA_CACHE  = "hal_openalex_uniques.csv"
OA_RESULT_COLS = ["hal_id", "in_openalex", "match_method",
                  "openalex_id", "openalex_title", "title_sim"]

try:
    oa_cache = pd.read_csv(
        OA_CACHE,
        usecols=lambda c: c in OA_RESULT_COLS,
        encoding="utf-8-sig",
        dtype={"in_openalex": "boolean"},
    )
    df_unique = df_unique.merge(oa_cache, on="hal_id", how="left")
    df_unique["in_openalex"] = df_unique["in_openalex"].fillna(False)
    n = df_unique["in_openalex"].sum()
    print(f"Résultats OpenAlex restaurés depuis '{OA_CACHE}' : {n} publications trouvées")
except FileNotFoundError:
    print(f"⚠  '{OA_CACHE}' introuvable — exécuter la section OpenAlex pour le générer.")
    df_unique["in_openalex"]    = False
    df_unique["match_method"]   = pd.NA
    df_unique["openalex_id"]    = pd.NA
    df_unique["openalex_title"] = pd.NA
    df_unique["title_sim"]      = pd.NA

# ── Propagation vers df (par labo) ───────────────────────────────────────────
OA_JOIN_COLS = ["hal_id", "in_openalex", "match_method", "openalex_id"]
df = df.drop(columns=[c for c in OA_JOIN_COLS[1:] if c in df.columns], errors="ignore")
df = df.merge(df_unique[OA_JOIN_COLS], on="hal_id", how="left")

print(f"df shape après enrichissement : {df.shape}")

---
# Croisement avec OpenAlex

Stratégie en deux passes sur `df_unique` :
1. **Passe DOI** : requête par lot (50 DOI à la fois) via `filter=doi:…`
2. **Passe titre** : pour les publications sans DOI (ou non trouvées par DOI), recherche sur `display_name.search` avec vérification de similarité (seuil 0.85)

> **Note** : la recherche par titre est heuristique — des faux positifs sont possibles sur des titres courts ou très génériques.
>
> Si les résultats OpenAlex ont déjà été calculés et exportés, **ne pas ré-exécuter les cellules de cette section** — les résultats sont restaurés automatiquement depuis le CSV.

In [ ]:
import difflib
import unicodedata
import re

# ── Paramètres OpenAlex ──────────────────────────────────────────────────────
OPENALEX_URL    = "https://api.openalex.org/works"
OPENALEX_MAILTO = "votre.email@example.com"   # remplacer pour le « polite pool »
OA_BATCH_SIZE   = 50     # max DOI par requête (recommandé par OpenAlex)
OA_SLEEP_SEC    = 0.1    # pause entre requêtes
TITLE_SIM_MIN   = 0.85   # seuil de similarité pour la correspondance par titre


def _norm_title(title: str) -> str:
    """Normalise un titre pour la comparaison : minuscules, sans accents ni ponctuations."""
    if not title:
        return ""
    t = unicodedata.normalize("NFD", title.lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _norm_doi(doi: str) -> str:
    """Renvoie le DOI sans le préfixe URI, en minuscules."""
    if not doi:
        return ""
    doi = doi.strip().lower()
    for prefix in ("https://doi.org/", "http://doi.org/", "doi:"):
        if doi.startswith(prefix):
            doi = doi[len(prefix):]
    return doi


def _oa_params(**extra) -> dict:
    return {"mailto": OPENALEX_MAILTO, "select": "id,doi,display_name", **extra}


def lookup_dois(doi_list: list[str]) -> dict[str, dict]:
    """Recherche par lots de DOIs. Renvoie {doi_normalisé -> {...}}."""
    results = {}
    for i in range(0, len(doi_list), OA_BATCH_SIZE):
        batch = doi_list[i : i + OA_BATCH_SIZE]
        filter_val = "|".join(batch)
        params = _oa_params(filter=f"doi:{filter_val}", per_page=OA_BATCH_SIZE)
        try:
            resp = requests.get(OPENALEX_URL, params=params, timeout=30)
            resp.raise_for_status()
            for work in resp.json().get("results", []):
                raw_doi = work.get("doi") or ""
                nd = _norm_doi(raw_doi)
                if nd:
                    results[nd] = {
                        "openalex_id":    work.get("id"),
                        "openalex_title": work.get("display_name"),
                    }
        except Exception as e:
            print(f"  ⚠ Erreur batch DOI [{i}:{i+OA_BATCH_SIZE}] : {e}")
        time.sleep(OA_SLEEP_SEC)
    return results


def lookup_title(title: str) -> dict | None:
    """Recherche par titre avec vérification de similarité."""
    if not title or len(title.strip()) < 10:
        return None
    search_title = title[:200]
    params = _oa_params(**{"filter": f'display_name.search:"{search_title}"', "per_page": 1})
    try:
        resp = requests.get(OPENALEX_URL, params=params, timeout=30)
        resp.raise_for_status()
        results = resp.json().get("results", [])
        if not results:
            return None
        work = results[0]
        oa_title = work.get("display_name") or ""
        sim = difflib.SequenceMatcher(
            None, _norm_title(title), _norm_title(oa_title)
        ).ratio()
        if sim >= TITLE_SIM_MIN:
            return {
                "openalex_id":    work.get("id"),
                "openalex_title": oa_title,
                "title_sim":      round(sim, 3),
            }
    except Exception as e:
        print(f"  ⚠ Erreur titre : {e}")
    return None

In [ ]:
# ── Initialisation des colonnes résultats ────────────────────────────────────
df_unique = df_unique.copy()
df_unique["in_openalex"]    = False
df_unique["match_method"]   = pd.NA
df_unique["openalex_id"]    = pd.NA
df_unique["openalex_title"] = pd.NA
df_unique["title_sim"]      = pd.NA

# ── Passe 1 : DOI ────────────────────────────────────────────────────────────
has_doi = df_unique["doi"].notna() & (df_unique["doi"].str.strip() != "")
dois_norm = df_unique.loc[has_doi, "doi"].apply(_norm_doi).tolist()
doi_index = df_unique.loc[has_doi].index

print(f"Publications avec DOI : {has_doi.sum()} / {len(df_unique)}")
print("Recherche par DOI dans OpenAlex…")

doi_results = lookup_dois(dois_norm)
print(f"  → {len(doi_results)} DOI trouvés dans OpenAlex")

for idx, doi_raw in zip(doi_index, dois_norm):
    if doi_raw in doi_results:
        hit = doi_results[doi_raw]
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "doi"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]

print(f"Publications trouvées via DOI : {(df_unique['match_method'] == 'doi').sum()}")

In [ ]:
# ── Passe 2 : titre ──────────────────────────────────────────────────────────
to_search_by_title = df_unique[
    (~df_unique["in_openalex"]) & (df_unique["titre"].notna())
].index

print(f"Publications à rechercher par titre : {len(to_search_by_title)}")
print("Recherche par titre dans OpenAlex (peut être long)…")

for idx in tqdm(to_search_by_title, desc="Recherche titre"):
    titre = df_unique.at[idx, "titre"]
    hit = lookup_title(titre)
    if hit:
        df_unique.at[idx, "in_openalex"]    = True
        df_unique.at[idx, "match_method"]   = "titre"
        df_unique.at[idx, "openalex_id"]    = hit["openalex_id"]
        df_unique.at[idx, "openalex_title"] = hit["openalex_title"]
        df_unique.at[idx, "title_sim"]      = hit["title_sim"]
    time.sleep(OA_SLEEP_SEC)

print(f"Publications trouvées via titre : {(df_unique['match_method'] == 'titre').sum()}")
print(f"Publications non trouvées : {(~df_unique['in_openalex']).sum()}")

## Taux de couverture global (df_unique)

In [ ]:
total   = len(df_unique)
found   = df_unique["in_openalex"].sum()
by_doi  = (df_unique["match_method"] == "doi").sum()
by_titl = (df_unique["match_method"] == "titre").sum()

print("=" * 50)
print(f"Publications HAL uniques           : {total:>7}")
print(f"Présentes dans OpenAlex            : {found:>7}  ({100*found/total:.1f}%)")
print(f"  dont trouvées par DOI            : {by_doi:>7}  ({100*by_doi/total:.1f}%)")
print(f"  dont trouvées par titre          : {by_titl:>7}  ({100*by_titl/total:.1f}%)")
print(f"Non trouvées dans OpenAlex         : {total-found:>7}  ({100*(total-found)/total:.1f}%)")
print("=" * 50)

df_unique.groupby("match_method", dropna=False).size().rename("count").to_frame()

## Taux de couverture par laboratoire

In [ ]:
# On repart de df (labo × publication) et on joint df_unique (résultats OA)
# pour être sûr d'avoir les bonnes valeurs de match_method et in_openalex.
df_labo = (
    df[["laboratoire", "hal_id", "doi"]]
    .merge(
        df_unique[["hal_id", "in_openalex", "match_method"]],
        on="hal_id", how="left"
    )
    .assign(_has_doi=lambda d: d["doi"].notna() & (d["doi"].str.strip() != ""))
)

stats_labo = (
    df_labo
    .groupby("laboratoire")
    .agg(
        n_hal      =("hal_id",       "count"),
        n_openalex =("in_openalex",  "sum"),
        n_doi      =("match_method", lambda x: (x == "doi").sum()),
        n_titre    =("match_method", lambda x: (x == "titre").sum()),
        n_avec_doi =("_has_doi",     "sum"),
        n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
    )
    .assign(
        taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
        taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
        taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
    )
    .sort_values("taux_global", ascending=False)
)

# Vérification : le total n_titre doit être >= (df_unique['match_method']=='titre').sum()
print(f"Total n_titre dans stats_labo : {stats_labo['n_titre'].sum()}")
print(f"Référence df_unique           : {(df_unique['match_method'] == 'titre').sum()}")
stats_labo

---
# Analyses de couverture par dimension

Les analyses suivantes utilisent `df_unique` (publications dédoublonnées) pour éviter de compter plusieurs fois une même publication affectée à plusieurs laboratoires.

## Couverture par année de publication

In [ ]:
def coverage_table(df_: pd.DataFrame, col: str) -> pd.DataFrame:
    """Calcule le taux de couverture OpenAlex pour chaque valeur d'une colonne.
    - taux_global : % de publications HAL trouvées dans OpenAlex (/ n_hal)
    - taux_doi    : parmi les pubs HAL avec DOI, % retrouvées dans OpenAlex (/ n_avec_doi)
    - taux_titre  : parmi les pubs HAL sans DOI, % retrouvées dans OpenAlex (/ n_sans_doi)
    """
    has_doi = df_["doi"].notna() & (df_["doi"].str.strip() != "")
    return (
        df_.assign(_has_doi=has_doi)
        .groupby(col, dropna=False)
        .agg(
            n_hal      =("hal_id",       "count"),
            n_openalex =("in_openalex",  "sum"),
            n_doi      =("match_method", lambda x: (x == "doi").sum()),
            n_titre    =("match_method", lambda x: (x == "titre").sum()),
            n_avec_doi =("_has_doi",     "sum"),
            n_sans_doi =("_has_doi",     lambda x: (~x).sum()),
        )
        .assign(
            taux_global=lambda d: (d["n_openalex"] / d["n_hal"]      * 100).round(1),
            taux_doi   =lambda d: (d["n_doi"]      / d["n_avec_doi"] * 100).round(1),
            taux_titre =lambda d: (d["n_titre"]    / d["n_sans_doi"] * 100).round(1),
        )
        .sort_index()
    )


stats_annee = coverage_table(df_unique, "annee")
print("Couverture OpenAlex par année de publication")
stats_annee

## Couverture par type de document (docType_s)

In [ ]:
stats_type = (
    coverage_table(df_unique, "type_doc")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par type de document")
stats_type

## Couverture par langue de publication (language_s)

In [ ]:
stats_langue = (
    coverage_table(df_unique, "langue")
    .sort_values("taux_global", ascending=False)
)
print("Couverture OpenAlex par langue de publication")
stats_langue

---
## Export final

In [ ]:
df_unique.to_csv("hal_openalex_uniques.csv",          index=False, encoding="utf-8-sig")
df.to_csv(       "hal_openalex_par_labo.csv",         index=False, encoding="utf-8-sig")
stats_labo.to_csv(  "couverture_par_labo.csv",                     encoding="utf-8-sig")
stats_annee.to_csv( "couverture_par_annee.csv",                    encoding="utf-8-sig")
stats_type.to_csv(  "couverture_par_type_doc.csv",                 encoding="utf-8-sig")
stats_langue.to_csv("couverture_par_langue.csv",                   encoding="utf-8-sig")

print("Fichiers exportés :")
print("  hal_openalex_uniques.csv      (df_unique enrichi)")
print("  hal_openalex_par_labo.csv     (df non dédoublonné enrichi)")
print("  couverture_par_labo.csv")
print("  couverture_par_annee.csv")
print("  couverture_par_type_doc.csv")
print("  couverture_par_langue.csv")